# Question 7
Compare all four ingestion patterns covered (batch CTAS, COPY INTO, Autoloader, Lakeflow Declarative Pipelines) on cost, latency, and operational complexity, and recommend which one Cyntexa should use for a file source that arrives unpredictably throughout the day. 


## Comparison Matrix

| **Ingestion Pattern**                     | **Cost**                                                                                                          | **Latency**                                                                               | **Operational Complexity**                                                                                            |
| ----------------------------------------- | ----------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------- |
| **Batch CTAS** (`CREATE TABLE AS SELECT`) | **High** — Re-computes the entire dataset on each run, resulting in redundant compute.                            | **High** — Relies on scheduled batches and is not suitable for near-real-time processing. | **High** — Requires manual orchestration, state management, and file tracking.                                        |
| **COPY INTO**                             | **Low–Medium** — Processes only new files and avoids reprocessing already loaded files.                           | **Medium** — Depends on scheduled polling and file-scanning overhead at scale.            | **Medium** — Requires manual scheduling and file-path management.                                                     |
| **Auto Loader** (`cloudFiles`)            | **Low** — Efficiently performs incremental file ingestion using file notifications or directory listing.          | **Low** — Supports near-real-time streaming or triggered execution.                       | **Low** — Automatically manages incremental state, file tracking, and schema evolution.                               |
| **Lakeflow Declarative Pipelines**        | **Low–Medium** — Uses optimized incremental processing and can simplify resource scaling with serverless options. | **Low** — Supports continuous streaming and triggered execution.                          | **Very Low** — Provides a managed declarative framework with built-in data quality, retries, and pipeline management. |

## Recommendation for Cyntexa

Cyntexa should use **Auto Loader inside a Lakeflow Declarative Pipeline**, preferably with **Triggered mode (`AvailableNow`)**.

### Why?

* **Cost Optimization:** Files arrive unpredictably throughout the day. Running a continuously active streaming cluster could waste compute when no new files are available. `AvailableNow` processes the currently available files and then stops the compute.

* **Incremental Processing:** Auto Loader tracks which files have already been processed, so only new files are ingested instead of repeatedly scanning and processing the entire dataset.

* **Low Latency:** When the pipeline is triggered periodically or based on file-arrival events, new files can be processed shortly after they arrive without maintaining a continuously running cluster.

* **Low Maintenance:** Lakeflow Declarative Pipelines manages pipeline execution, incremental processing, retries, and schema evolution, reducing the amount of custom orchestration code.

### Final Recommendation

**Auto Loader + Lakeflow Declarative Pipelines + `AvailableNow`** provides the best balance of **cost, latency, scalability, and operational simplicity** for Cyntexa's unpredictable file-arrival pattern.


# Question 8

 Design a recovery runbook: if a bad file corrupts the silver table at 2am, walk through the exact commands (DESCRIBE HISTORY, RESTORE or time travel + overwrite) an on-call engineer would run. 


# Production Recovery Runbook: Silver Table Data Corruption

**Severity:** Critical (P1/P2)
**Target Component:** Delta Lake / Databricks Silver Layer
**Objective:** Revert corrupt commits injected at approximately 2:00 AM using Delta Lake Time Travel and Restore capabilities while preserving audit trails.

## Step 1: Immediate Triage & Scope Isolation

1. **Pause Upstream Pipelines:** Immediately pause all incoming ingestion jobs (Auto Loader / Structured Streaming / Airflow DAGs) writing to the corrupted Silver table to prevent further bad data from being appended.

2. **Inspect Commit History:** Retrieve the Delta table transaction history to identify the exact `version` number and timestamp where the corruption occurred.

```sql
-- Identify corrupt operations and the bad job run around 02:00 AM
DESCRIBE HISTORY silver.user_events;
```

## Step 2: Time Travel Data Verification

Before executing the rollback, inspect both the corrupt and historical states to verify the target rollback version. For example, if `Version 42` is corrupt, `Version 41` may be the last known good version.

```sql
-- Check corrupt state (Version 42)
SELECT count(*), count_if(user_id IS NULL)
FROM silver.user_events VERSION AS OF 42;

-- Validate last known good state (Version 41)
SELECT count(*), count_if(user_id IS NULL)
FROM silver.user_events VERSION AS OF 41;
```

## Step 3: Table Restoration Execution

### Native Delta RESTORE

Use the native Delta `RESTORE` command to revert the table to the last known good version.

```sql
-- Restore by exact version number
RESTORE TABLE silver.user_events TO VERSION AS OF 41;

-- Alternative: Restore by timestamp before the corrupt run
RESTORE TABLE silver.user_events
TO TIMESTAMP AS OF '2026-08-27 01:55:00';
```

The restore operation creates a new transaction commit while preserving the existing transaction history, providing an audit trail of the recovery operation.

## Step 4: Post-Restore Validation & Downstream Recovery

1. **Verify State Integrity:** Check the restored table's row count and confirm that the restore operation was recorded in the transaction history.

```sql
-- Check overall row count
SELECT count(*)
FROM silver.user_events;

-- Verify the restore operation was recorded
DESCRIBE HISTORY silver.user_events;
```

2. **Downstream Backfill:** Trigger targeted re-runs for downstream Gold tables and aggregations for the affected time window between 02:00 AM and the current timestamp.

3. **Resume Services:** Re-enable the paused ingestion jobs and monitor data quality checks and pipeline execution to ensure the corruption does not recur.

## Recovery Summary

The recommended recovery approach is:

**Pause ingestion → Inspect Delta history → Validate last known good version using Time Travel → RESTORE the Silver table → Validate restored state → Backfill downstream Gold tables → Resume ingestion.**

This approach minimizes data loss, avoids unnecessary full re-ingestion, and preserves the Delta transaction history needed for auditing and incident investigation.


# Question 9


In [0]:
%sql
-- Step 1: Ek dummy table banao
CREATE TABLE IF NOT EXISTS cyntexa_dev.day_7.test_sla_table (id INT, status STRING) USING DELTA;

-- Step 2: Continuous batch updates run karo (simulated commits)
INSERT INTO cyntexa_dev.day_7.test_sla_table VALUES (1, 'active');
-- 


In [0]:
%sql
-- (WAIT 1 MINUTE)
INSERT INTO cyntexa_dev.day_7.test_sla_table VALUES (2, 'pending');


In [0]:
%sql
-- (WAIT 2 MINUTES)
UPDATE cyntexa_dev.day_7.test_sla_table SET status = 'completed' WHERE id = 1;

-- Step 3: Ab DESCRIBE HISTORY query run karke interval check karo
DESCRIBE HISTORY cyntexa_dev.day_7.test_sla_table;

In [0]:
%sql
-- Step 3: Ab DESCRIBE HISTORY query run karke interval check karo
DESCRIBE HISTORY cyntexa_dev.day_7.test_sla_table;

In [0]:
%sql

WITH history_data AS (
  SELECT 
    version,
    timestamp,
    operation,
    LAG(timestamp) OVER (ORDER BY timestamp ASC) AS prev_timestamp
  FROM (DESCRIBE HISTORY cyntexa_dev.day_7.test_sla_table)
  WHERE operation IN ('WRITE', 'UPDATE', 'MERGE')
)
SELECT 
  version,
  operation,
  timestamp AS current_update_time,
  prev_timestamp AS previous_update_time,
  -- Seconds / Minutes between updates calculate karne ke liye:
  ROUND((CAST(timestamp AS LONG) - CAST(prev_timestamp AS LONG)), 2) AS seconds_since_last_update,
  ROUND((CAST(timestamp AS LONG) - CAST(prev_timestamp AS LONG)) / 60.0, 2) AS minutes_since_last_update
FROM history_data
ORDER BY version DESC;

1. Environment Setup (Dummy Table Creation)

Standard Delta Lake table create ki (CREATE TABLE USING DELTA) taaki Delta transaction logging automatically enable ho jaye.

2. Simulation of Workload (Commits over Time)

Simulated batch write operations run kiye:

Pehla INSERT run karke base version 0 generate kiya.

Time interval pause dekar second INSERT (version 1) execute kiya.

Further delay ke baad UPDATE statement run kiya (version 2).

Purpose: Multi-version commit history generate karna dynamic intervals ke saath.

3. Inspection of Transaction Log

Raw metadata inspect karne ke liye DESCRIBE HISTORY query execute ki.

Commit log check kiya—jahan har commit record ke saath specific timestamp, operation type, and version tag assign hua.

4. Analytical Reporting (Data Freshness Calculation)

Analytical Window Function (LAG()) use karke previous timestamp capture kiya.

Current timestamp se difference calculate karke interval (seconds/minutes) derive kiye.

Operational activities filter karke actual update frequency baseline ready ki, jisse SLA compliance validate hoti hai.